## Import Statements

In [31]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [32]:
# Download the csv file and upload to your google drive.
# Store it in a folder named data, if you want to match the path

# This allows colab to access your drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Load Data

In [33]:
# This path assumes you stored the csv in a file named 'data' under your mydrive
data = pd.read_csv('/content/drive/MyDrive/data/heart_2022_with_nans.csv')
data.head()


,State,Sex,GeneralHealth,PhysicalHealthDays,MentalHealthDays,LastCheckupTime,PhysicalActivities,SleepHours,RemovedTeeth,HadHeartAttack,...,HeightInMeters,WeightInKilograms,BMI,AlcoholDrinkers,HIVTesting,FluVaxLast12,PneumoVaxEver,TetanusLast10Tdap,HighRiskLastYear,CovidPos
0,Alabama,Female,Very good,0.0,0.0,Within past year (anytime less than 12 months ...,No,8.0,NaN,No,...,NaN,NaN,NaN,No,No,Yes,No,"Yes, received tetanus shot but not sure what type",No,No
1,Alabama,Female,Excellent,0.0,0.0,NaN,No,6.0,NaN,No,...,1.60,68.04,26.57,No,No,No,No,"No, did not receive any tetanus shot in the pa...",No,No
2,Alabama,Female,Very good,2.0,3.0,Within past year (anytime less than 12 months ...,Yes,5.0,NaN,No,...,1.57,63.50,25.61,No,No,No,No,NaN,No,Yes
3,Alabama,Female,Excellent,0.0,0.0,Within past year (anytime less than 12 months ...,Yes,7.0,NaN,No,...,1.65,63.50,23.30,No,No,Yes,Yes,"No, did not receive any tetanus shot in the pa...",No,No
4,Alabama,Female,Fair,2.0,0.0,Within past year (anytime less than 12 months ...,Yes,9.0,NaN,No,...,1.57,53.98,21.77,Yes,No,No,Yes,"No, did not receive any tetanus shot in the pa...",No,No


In [34]:
data.shape

(445132, 40)

In [35]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 445132 entries, 0 to 445131
Data columns (total 40 columns):
 #   Column                     Non-Null Count   Dtype  
---  ------                     --------------   -----  
 0   State                      445132 non-null  object 
 1   Sex                        445132 non-null  object 
 2   GeneralHealth              443934 non-null  object 
 3   PhysicalHealthDays         434205 non-null  float64
 4   MentalHealthDays           436065 non-null  float64
 5   LastCheckupTime            436824 non-null  object 
 6   PhysicalActivities         444039 non-null  object 
 7   SleepHours                 439679 non-null  float64
 8   RemovedTeeth               433772 non-null  object 
 9   HadHeartAttack             442067 non-null  object 
 10  HadAngina                  440727 non-null  object 
 11  HadStroke                  443575 non-null  object 
 12  HadAsthma                  443359 non-null  object 
 13  HadSkinCancer              44

## Data Cleaning

This section cleans the raw dataset for analysis and saves it to a new csv file.

### Drop Missing Target Values

`HadHeartAttack`, and `HadAngina` are our target variables. Rows where these is missing are dropped from the dataset.

In [36]:
# Dropping all the rows where the HadHeartAttack or HadAngina columns are empty
data = data.dropna(subset=['HadHeartAttack', 'HadAngina'])

### Fill Numeric Columns

Missing numeric values are filled with the column median.

In [37]:
numeric_cols = ['PhysicalHealthDays', 'MentalHealthDays', 'SleepHours', 'HeightInMeters', 'WeightInKilograms', 'BMI']

# Setting the NaNs with the median value for each column
data[numeric_cols] = data[numeric_cols].fillna(data[numeric_cols].median())

### Fill Categorical Columns

Missing categorical values are filled with the 'Unknown'.

In [38]:
# Grabbing all rows whos datatype is object (basically all categorical columns)
categorical_cols = data.select_dtypes(include='object').columns.tolist()

# Filling the empty values for the columns with "Unknown"
data[categorical_cols] = data[categorical_cols].fillna('Unknown')

### Adding Heart Disease Column

A participant is considered to have heart disease if they either `HadHeartAttack` or `HadAngina`. So we are adding a new column that checks for one of the other and is true if one or the other is yes.

In [39]:
# A person has heart disease if they had a heart attack OR angina
# 1 if yes or 0 if not
data['HadHeartDisease'] = ((data['HadHeartAttack'] == 'Yes') | (data['HadAngina'] == 'Yes')).astype(int)

### Save The Cleaned Data
Saving the cleaned data to a new csv

In [40]:
# Saving to data folder
data.to_csv('/content/drive/MyDrive/data/heart_cleaned.csv', index=False)